# Imports

In [ ]:
import numpy as np
import torch
import datasets
import transformers
import bitsandbytes as bnb
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig, EarlyStoppingCallback
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from accelerate import cpu_offload
from utils import get_optimal_training_config, get_vm_usage_metrics
import wandb
from datetime import datetime
import random
import math
from tqdm.auto import tqdm
import os
import gc

import _config

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

os.environ["WANDB_API_KEY"] = _config.WANDB_API_KEY
os.environ["WANDB_PROJECT"] = _config.WANDB_PROJECT

In [ ]:
training_config = get_optimal_training_config()

# Enable TF32 for Ampere+ GPUs
if training_config.get('tf32', False):
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# Data

In [ ]:
ds = datasets.load_dataset('gretelai/synthetic_text_to_sql', streaming=False)
ds_train, ds_test = ds['train'], ds['test']

split = ds_train.train_test_split(test_size=0.025, seed=42)
ds_train = split['train']
ds_valid = split['test']

ds_train

# Model

In [ ]:
checkpoint = "Qwen/Qwen3-0.6B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # change to bfloat if available
)

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    device_map="auto",
    attn_implementation=training_config["attn_implementation"],
    quantization_config=bnb_config,
    # dtype=torch.bfloat16 # not specifying on T4 to avoid unscaling errors
)

model.config.use_cache = False
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()

get_vm_usage_metrics()
# model = cpu_offload(model)

# Formatting functions

In [ ]:
# used for training
def construct_message_with_assistant_content(example):
    messages = [
        {"role": "system", "content": f"The user asks a question. Your task is to generate the SQL query to answer that question. Return SQL query only. The context of the question is the following: '{example['sql_context']}'"},
        {"role": "user", "content": example['sql_prompt']},
        {"role": "assistant", "content": example['sql']}
    ]
    return {'messages': messages}

In [ ]:
ENABLE_THINKING = False

def formatting_func(example, enable_thinking=ENABLE_THINKING):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False, # no generation prompt during training
        enable_thinking=ENABLE_THINKING
    )

# Data preprocessing

In [ ]:
TRAIN_SIZE = 4096
VALID_SIZE = 1024

ds_train_sample = ds_train.take(TRAIN_SIZE)
ds_valid_sample = ds_valid.take(VALID_SIZE)

print(len(ds_train_sample), len(ds_valid_sample), len(ds_test))

ds_train_with_assistant_content = ds_train_sample.map(construct_message_with_assistant_content)
ds_valid_with_assistant_content = ds_valid_sample.map(construct_message_with_assistant_content)

# Hyperparameter optimization

In [ ]:
# --- sweep definition ---
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

sweep_config = {
    'name': f'-sweep-sft-lora-{timestamp}',
    'method': 'bayes',
    'metric': {
        'name': 'eval_loss',
        'goal': 'minimize'
    },
    'parameters': {
        'optimizer':            {'values': ['adam', 'adamw', 'nadam', 'adamax']},
        'effective_batch_size': {'values': [16, 32, 64, 128, 256, 512]},
        'learning_rate':        {'values': [1e-4, 5e-5, 1e-5, 5e-6, 1e-6]},
        'lr_scheduler_type':    {'values': ['cosine', 'linear', 'cosine_with_restarts', 'constant_with_warmup']},
        'weight_decay':         {'values': [0.0, 0.01, 0.1]},
        'neftune_noise_alpha':  {'values': [0.0, 0.01, 0.1]},
        'betas':                {'values': [(0.9, 0.999), (0.95, 0.999), (0.9, 0.9999)]},
        'warmup_ratio':         {'values': [0.05, 0.1, 0.2]},
        'epochs':               {'values': [1]},
        'lora_r':               {'values': [4, 8, 16, 32]},
        'lora_alpha':           {'values': [2, 4, 8, 16, 32, 64]},
        'lora_dropout':         {'values': [0.01, 0.05, 0.1, 0.2]}
    }
}

sweep_id = wandb.sweep(sweep_config, project=os.environ["WANDB_PROJECT"])
# sweep_id = 'je9m6in0'  # continue a crashed sweep

# OUTPUT_DIR = f"./sft-sweep-output/{run.name}" # outputs every run in a different dir
OUTPUT_DIR = f"./sft-sweep-output" # overwrites every run
PER_DEVICE_BATCH_SIZE = 1
EPSILON = 1e-6  # 1e-6 is safer than default 1e-8 for bf16/fp16 training
NUM_EVALS_PER_RUN = 3  # target number of evals per run; should be at least 1
seed = 42

# --- optimizer map ---
if training_config["fp16"]:
    optimizer_map = {
        "adam":   torch.optim.Adam,
        "adamw":  torch.optim.AdamW,
        "nadam":  torch.optim.NAdam,
        "adamax": torch.optim.Adamax,
    }
else:
    optimizer_map = {
        "adam":   bnb.optim.Adam8bit,
        "adamw":  bnb.optim.AdamW8bit,
        "nadam":  torch.optim.NAdam,
        "adamax": torch.optim.Adamax,
    }

# --- sweep train function ---
def sweep_train():
    with wandb.init() as run:
        config = wandb.config

        # seed everything
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        transformers.set_seed(seed)
        # optional: makes CUDA ops deterministic (slower, but fully reproducible)
        # torch.backends.cudnn.deterministic = True
        # torch.backends.cudnn.benchmark = False

        gradient_accumulation_steps = max(1, config.effective_batch_size // PER_DEVICE_BATCH_SIZE)

        dataset_size = len(ds_train_with_assistant_content)
        steps_per_epoch = max(1, dataset_size // (PER_DEVICE_BATCH_SIZE * gradient_accumulation_steps))
        total_steps = steps_per_epoch * config.epochs

        # spread evals evenly across training; clamp so eval_steps <= total_steps
        # guaranteeing at least one eval even with very large batches / small datasets
        eval_steps = max(1, min(total_steps, total_steps // NUM_EVALS_PER_RUN))
        logging_steps = max(1, eval_steps // 3)  # log more frequently than eval

        training_args = TrainingArguments(
            output_dir=OUTPUT_DIR,
            per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
            gradient_accumulation_steps=gradient_accumulation_steps,
            learning_rate=config.learning_rate,
            lr_scheduler_type=config.lr_scheduler_type,
            warmup_ratio=config.warmup_ratio,
            num_train_epochs=config.epochs,
            weight_decay=config.weight_decay,
            neftune_noise_alpha=config.neftune_noise_alpha if config.neftune_noise_alpha > 0 else None,
            max_grad_norm=1,

            use_liger_kernel=training_config['liger_kernel'],
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},

            # dynamic precision based on GPU
            bf16=training_config['bf16'],
            fp16=training_config['fp16'],
            bf16_full_eval=training_config['bf16'],
            fp16_full_eval=training_config['fp16'],
            tf32=training_config.get('tf32', False),

            save_strategy="steps",
            save_steps=eval_steps,
            save_total_limit=2,
            eval_strategy="steps",
            eval_steps=eval_steps,
            logging_strategy="steps",
            logging_steps=logging_steps,

            report_to=['wandb'],
            run_name=run.name,

            metric_for_best_model="eval_loss",
            greater_is_better=False,
            load_best_model_at_end=True,
            seed=seed
        )

        def build_optimizer(model):
            optimizer_class = optimizer_map[config.optimizer]
            return optimizer_class(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay,
                betas=config.betas,
                eps=EPSILON,
            )


        
        peft_config = LoraConfig(
            r=config.lora_r,
            lora_alpha=config.lora_alpha,
            lora_dropout=config.lora_dropout,
            bias="none",
            task_type="CAUSAL_LM"
        )
        model.requires_grad_(False)                     # freeze base weights (precautionary)
        model_peft = get_peft_model(model, peft_config) # inject a LoRA adapter

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )
        
        trainer = SFTTrainer(
            model=model_peft,
            train_dataset=ds_train_with_assistant_content,
            eval_dataset=ds_valid_with_assistant_content,
            formatting_func=formatting_func,
            args=training_args,
            optimizers=(build_optimizer(model_peft), None),  # (optimizer, scheduler)
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )

        # training setup summary
        warmup_steps = int(total_steps * config.warmup_ratio)

        print("===== Training Setup Summary =====")
        print(f"Num epochs:            {config.epochs}")
        print(f"Effective batch size:  {config.effective_batch_size}")
        print(f"Per-device batch size: {PER_DEVICE_BATCH_SIZE}")
        print(f"Gradient accumulation: {gradient_accumulation_steps}")
        print(f"LR scheduler:          {config.lr_scheduler_type}")
        print(f"Epsilon:               {EPSILON}")
        print(f"Dataset size:          {dataset_size}")
        print(f"Steps per epoch:       {steps_per_epoch}")
        print(f"Total training steps:  {total_steps}")
        print(f"Warmup steps:          {warmup_steps}")
        print(f"Eval steps:            {eval_steps}  (target {NUM_EVALS_PER_RUN} evals/run)")
        print(f"Logging steps:         {logging_steps}")
        print("===================================")
        print(f"Start time: {datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}")

        trainer.train()

        print(f"End time: {datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}")

        # WandB logging of eval metrics
        for log in trainer.state.log_history:
            if 'eval_loss' in log:
                wandb.log({
                    "eval_loss":            log['eval_loss'],
                    "eval_perplexity":      math.exp(log['eval_loss']),
                    "step":                 log['step'],
                    "learning_rate":        config.learning_rate,
                    "lr_scheduler_type":    config.lr_scheduler_type,
                    "weight_decay":         config.weight_decay,
                    "neftune_noise_alpha":  config.neftune_noise_alpha,
                    "betas":                config.betas,
                    "warmup_ratio":         config.warmup_ratio,
                    "effective_batch_size": config.effective_batch_size,
                    "optimizer":            config.optimizer,
                    "epsilon":              EPSILON,
                    "seed":                 seed,
                })

        del trainer
        torch.cuda.empty_cache()
        gc.collect()

wandb.agent(sweep_id, function=sweep_train, count=20)

# Final model training

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# --- W&B init ---
# starting a new run
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
RUN_NAME = f'sft-final-model-{timestamp}'
wandb.init(
    project=os.environ["WANDB_PROJECT"],
    name=RUN_NAME
)
RESUME_TRAINING = False

# # resuming the prev run
# timestamp = '2025-08-19_08-33-30'
# RUN_NAME = f'sft-final-model-{timestamp}'
# run_id = 'imoh6jtd'
# wandb.init(
#     project=os.environ["WANDB_PROJECT"],
#     name=RUN_NAME,
#     id=run_id,         # resume previous run if available
#     resume="allow",    # allows resuming crashed run
# )
# RESUME_TRAINING = True


# --- hyperparameters ---
OUTPUT_DIR = "./sft-final_model-output"
PER_DEVICE_BATCH_SIZE = 1
optimizer        = 'nadam'
effective_batch_size = 32
learning_rate    = 1e-4
lr_scheduler_type = "cosine"
weight_decay     = 0.0
neftune_noise_alpha = 0.0
betas            = (0.95, 0.999)
warmup_ratio     = 0.05
epochs           = 1
EPSILON          = 1e-6  # 1e-6 is safer than default 1e-8 for bf16/fp16 training
NUM_EVALS_PER_RUN = 32   # target number of evals per run; guarantees at least 1
lora_r = 16
lora_alpha = 64
lora_dropout = 0.01
seed = 42

gradient_accumulation_steps = max(1, effective_batch_size // PER_DEVICE_BATCH_SIZE)

# --- compute steps BEFORE TrainingArguments so eval_steps is always valid ---
dataset_size = len(ds_train_with_assistant_content)
steps_per_epoch = max(1, dataset_size // (PER_DEVICE_BATCH_SIZE * gradient_accumulation_steps))
total_steps = steps_per_epoch * epochs

# Spread evals evenly across training; clamp so eval_steps <= total_steps,
# guaranteeing at least one eval even with very large batches / small datasets
eval_steps = max(1, min(total_steps, total_steps // NUM_EVALS_PER_RUN))


# --- seed everything
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
transformers.set_seed(seed)

# --- optimizer map ---
if training_config["fp16"]:
    optimizer_map = {
        "adam":   torch.optim.Adam,
        "adamw":  torch.optim.AdamW,
        "nadam":  torch.optim.NAdam,
        "adamax": torch.optim.Adamax,
    }
else:
    optimizer_map = {
        "adam":   bnb.optim.Adam8bit,
        "adamw":  bnb.optim.AdamW8bit,
        "nadam":  torch.optim.NAdam,
        "adamax": torch.optim.Adamax,
    }

# --- training args ---
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    lr_scheduler_type=lr_scheduler_type,
    warmup_ratio=warmup_ratio,
    num_train_epochs=epochs,
    weight_decay=weight_decay,
    neftune_noise_alpha=neftune_noise_alpha if neftune_noise_alpha > 0 else None,
    max_grad_norm=1,

    use_liger_kernel=training_config['liger_kernel'], # available on Ampere and Hopper
    gradient_checkpointing=True, # ~ -30% throughput for ~3–4× memory reduction
    gradient_checkpointing_kwargs={"use_reentrant": False},
    
    # dynamic precision based on GPU
    bf16=training_config['bf16'],
    fp16=training_config['fp16'],
    bf16_full_eval=training_config['bf16'],
    fp16_full_eval=training_config['fp16'],
    tf32=training_config.get('tf32', False), # enable TF32 for Ampere+ (faster matmul with minimal precision loss)

    save_strategy="steps",
    save_steps=eval_steps,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=eval_steps,
    logging_strategy="steps",
    logging_steps=1, # log every step for full granularity in the final run

    report_to=['wandb'],
    run_name=RUN_NAME,

    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    seed=seed
)

# --- optimizer builder ---
def build_optimizer(model):
    optimizer_class = optimizer_map[optimizer]
    return optimizer_class(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
        betas=betas,
        eps=EPSILON,
    )

peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM"
)
model.requires_grad_(False)                     # freeze base weights (precautionary)
model_peft = get_peft_model(model, peft_config) # inject a LoRA adapter

# --- trainer ---
trainer = SFTTrainer(
    model=model_peft,
    train_dataset=ds_train_with_assistant_content,
    eval_dataset=ds_valid_with_assistant_content,
    formatting_func=formatting_func,
    args=training_args,
    optimizers=(build_optimizer(model_peft), None),  # (optimizer, scheduler)
    callbacks=[EarlyStoppingCallback(early_stopping_patience=25)]
)

# --- training setup summary ---
warmup_steps = int(total_steps * warmup_ratio)

print("===== Training Setup Summary =====")
print(f"Num epochs:            {epochs}")
print(f"Effective batch size:  {effective_batch_size}")
print(f"Per-device batch size: {PER_DEVICE_BATCH_SIZE}")
print(f"Gradient accumulation: {gradient_accumulation_steps}")
print(f"LR scheduler:          {lr_scheduler_type}")
print(f"Epsilon:               {EPSILON}")
print(f"Dataset size:          {dataset_size}")
print(f"Steps per epoch:       {steps_per_epoch}")
print(f"Total training steps:  {total_steps}")
print(f"Warmup steps:          {warmup_steps}")
print(f"Eval steps:            {eval_steps}  (target {NUM_EVALS_PER_RUN} evals/run)")
print(f"Logging steps:         {training_args.logging_steps}")
print("===================================")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}")

# --- training ---
last_checkpoint = None
if RESUME_TRAINING and os.path.isdir(OUTPUT_DIR):
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint is not None:
    print(f"Resuming training from checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting fresh training run")
    trainer.train()

print(f"End time: {datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}")

# --- WandB logging of eval metrics ---
for log in trainer.state.log_history:
    if 'eval_loss' in log:
        wandb.log({
            "eval_loss":            log['eval_loss'],
            "eval_perplexity":      math.exp(log['eval_loss']),
            "step":                 log['step'],
            "learning_rate":        learning_rate,
            "lr_scheduler_type":    lr_scheduler_type,
            "weight_decay":         weight_decay,
            "neftune_noise_alpha":  neftune_noise_alpha,
            "betas":                betas,
            "warmup_ratio":         warmup_ratio,
            "effective_batch_size": effective_batch_size,
            "optimizer":            optimizer,
            "epsilon":              EPSILON,
            "seed":                 seed
        })

wandb.finish()  # finish the run

In [ ]:
model_path = os.path.join(OUTPUT_DIR, 'best_model')
trainer.save_model(model_path)

# Evaluation

In [ ]:
def construct_message(prompt, context):
    return [
        {"role": "system", "content": f"The user asks a question. Your task is to generate the SQL query to answer that question. Return SQL query only. The context of the question is the following: '{context}'"},
        {"role": "user", "content": prompt}
    ]

def generate_model_response_batch(model, tokenizer, messages_list, enable_thinking=True, max_new_tokens=512):
    texts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking
        )
        for messages in messages_list
    ]

    model_inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        padding_side='left'
    ).to(model.device)

    model.eval()
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens
    )

    responses = []
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids):
        # Slice to get only generated part
        output_only_ids = output_ids[len(input_ids):].tolist()

        # Try to find `</think>` (id 151668)
        try:
            index = len(output_only_ids) - output_only_ids[::-1].index(151668)
        except ValueError:
            index = 0

        if enable_thinking:
            thinking_content = tokenizer.decode(
                output_only_ids[:index],
                skip_special_tokens=True
            ).strip("\n")
            content = tokenizer.decode(
                output_only_ids[index:],
                skip_special_tokens=True
            ).strip("\n")
        else:
            thinking_content = None
            content = tokenizer.decode(
                output_only_ids,
                skip_special_tokens=True
            ).strip("\n")

        responses.append({
            'thinking_content': thinking_content,
            'content': content
        })

    return responses

In [ ]:
torch.cuda.empty_cache()
get_vm_usage_metrics()

In [ ]:
model_path = './sft-final_model-output/best_model'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path).to('cuda')
model.eval()

In [ ]:
BATCH_SIZE = 512
ENABLE_THINKING = False
MAX_NEW_TOKENS = 512


prompts = [ds_test[id]['sql_prompt'] for id in range(len(ds_test))]
contexts = [ds_test[id]['sql_context'] for id in range(len(ds_test))]

responses = []
print(f"Start time: {datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}")
for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
    batch_prompts = prompts[i : i + BATCH_SIZE]
    batch_contexts = contexts[i : i + BATCH_SIZE]

    messages_list = [
        construct_message(prompt=p, context=c)
        for p, c in zip(batch_prompts, batch_contexts)
    ]

    batch_responses = generate_model_response_batch(model, tokenizer, messages_list, enable_thinking=ENABLE_THINKING, max_new_tokens=MAX_NEW_TOKENS)

    responses.extend(batch_responses)

print(f"End time: {datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}")